In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from sklearn.metrics import classification_report, confusion_matrix

sys.path.append("src")
from entropy_pruning import UNILoRAClassifier, build_loaders, evaluate_classifier, set_seed, train_classifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
CFG = dict(
    data_dir="/data/BREAKHIS",
    img_size=224,
    batch_size=8,
    num_workers=4,
    seed=42,
    epochs=20,
    lr_head=1e-3,
    lr_backbone=1e-5,
    weight_decay=0.01,
    label_smoothing=0.1,
)

set_seed(CFG["seed"])
dataset_name = Path(CFG["data_dir"]).name
output_dir = Path(f"checkpoints/{dataset_name}/uni_finetuned")
output_dir.mkdir(parents=True, exist_ok=True)
ckpt_path = output_dir / "best_model.pt"


In [ ]:
loaders = build_loaders(
    data_dir=CFG["data_dir"],
    img_size=CFG["img_size"],
    batch_size=CFG["batch_size"],
    num_workers=CFG["num_workers"],
)
print("Classi:", loaders.class_names)


In [ ]:
model = UNILoRAClassifier(loaders.n_classes).to(device)
train_out = train_classifier(
    model=model,
    train_loader=loaders.train_loader,
    val_loader=loaders.val_loader,
    device=device,
    epochs=CFG["epochs"],
    lr_backbone=CFG["lr_backbone"],
    lr_head=CFG["lr_head"],
    weight_decay=CFG["weight_decay"],
    label_smoothing=CFG["label_smoothing"],
    save_path=ckpt_path,
)
print("Best val acc:", train_out["best_val_acc"])


In [ ]:
history = train_out["history"]
xs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(xs, history["train_loss"], label="train")
axes[0].plot(xs, history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[1].plot(xs, history["train_acc"], label="train")
axes[1].plot(xs, history["val_acc"], label="val")
axes[1].set_title("Accuracy")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
model.load_state_dict(torch.load(ckpt_path, map_location=device))
metrics = evaluate_classifier(model, loaders.test_loader, device)
print(metrics)


In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in loaders.test_loader:
        preds = model(imgs.to(device)).argmax(1).cpu()
        all_preds.append(preds)
        all_labels.append(labels)
all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

print(classification_report(all_labels, all_preds, target_names=loaders.class_names))
cm = confusion_matrix(all_labels, all_preds, normalize="true")
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=loaders.class_names, yticklabels=loaders.class_names, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix (normalizzata)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
